# RFN-Nest fine-tuned on the Blender forest clips — v3

## Versions

Each version is its own notebook and writes its own checkpoint, history and videos (named after `METHOD`), so earlier
results are never overwritten. To try a new setup, copy the latest notebook to the next number instead of editing it.

| notebook | `METHOD` | change | result on the test clips |
|---|---|---|---|
| `rfn_nest_blender_v1.ipynb` | `rfn_nest_blender` | RFN blocks only, `ALPHA = 700`, 10 epochs | close to the published weights; slightly less contrast stretching at night |
| `rfn_nest_blender_v2.ipynb` | `rfn_nest_blender_v2` | `ALPHA = 2000`, decoder trained too, 5 epochs | best fog removal in daylight (07); banding and lost thermal hot spots at night and dusk (01, 10) |
| `rfn_nest_blender_v3.ipynb` | `rfn_nest_blender_v3` | as v2 but decoder frozen | no banding, hot spots kept: v2's damage came from training the decoder. Fog removal slightly better than v1 (best SSIM_ref / PSNR_ref on 01), far from v2's daylight gain. Validation detail term 742 -> 720 only (v2: 618) |

Fine-tunes RFN-Nest on part of `forest_batch_1`, then fuses the clips that were kept out of training.
Meant for a **Colab T4 GPU** (Runtime > Change runtime type > T4 GPU). It also runs locally from inside the repo.

**Split, by clip** (not by frame: neighbouring frames of one clip are near-duplicates, so a frame split would leak).
Each split covers night or twilight and daylight.

| split | clips | used for |
|---|---|---|
| train (6) | 02 night-dark, 03 predawn, 05 morning, 06 midday, 08 overcast, 09 golden | updating the weights |
| val (1) | 04 sunrise | picking the best epoch |
| test (3) | 01 night-moon, 07 afternoon, 10 dusk | output videos only, never seen in training |

**Before you run it:** put the dataset on Google Drive at `DRIVE_DATA`. A single `forest_batch_1.zip` is much faster
to copy into Colab than the folder of ~8,500 PNGs (a plain folder works too).

**What is fine-tuned:** the RFN fusion blocks and, with `TRAIN_DECODER = True`, the NestFuse decoder, both starting
from the published weights. The encoder stays frozen. The repo's `train_fusionnet.py` trains only the RFN blocks; it
cannot be imported as is (`utils.py` needs the removed `scipy.misc`), so the training loop is here, with the same loss:
`ALPHA * (1 - MS-SSIM(output, target)) + feature loss(fused features, W_IR * IR + W_VI * target)`.

**Why v2.** In the first run (`rfn_nest_blender`: RFN blocks only, `W_IR = 6`, 10 epochs)
only the feature term improved; the detail term, which compares the output with the fog-free frame, did not, and nothing
improved after epoch 4. This setup therefore raises `ALPHA` from 700 to 2000, so the detail term is about 80% of the
starting loss instead of about 65%, trains the decoder too (at a lower learning rate), runs 5 epochs, validates on more
frames and logs both validation terms. 
**Why v3.** v2 reached its goal on the training loss (validation detail term 742 -> 618, while v1's stayed flat), and on
the daylight test clip it removed fog best (SSIM_ref 0.539, against 0.49 for v1 and the published weights). But on the
night and dusk clips its output is banded and the small thermal hot spots are gone. v2 changed two things at once,
`ALPHA` and training the decoder, so v3 keeps `ALPHA = 2000` and freezes the decoder again (`TRAIN_DECODER = False`)
to see which of the two causes it. If v3 keeps the hot spots, the decoder was the cause; if not, the detail term
itself pulls the output towards the visible frame, which has no hot spots, and the loss needs a thermal term.

`W_IR` / `W_VI` stay at 6 / 3 on purpose. The published RFN produces features close to 6 x thermal + 3 x visible, so any
other mix first forces it to rescale its features: on validation crops the starting feature loss is 142 at 6 / 3,
244 at 5 / 4, 341 at 4.5 / 4.5 and 2350 at 3 / 3 (detail: 259). Rebalancing the weights would make the feature term
dominate even more, the opposite of the aim; `ALPHA` shifts the balance without that side effect.

Differences from the repo's training:
- `VI_TARGET = 'visible_clear'`: the loss target is the fog-free render instead of the hazy input, so the network is
  rewarded for recovering detail rather than for reproducing fog. The **input** is always `visible_haze` + `thermal_haze`.
  Set `VI_TARGET = 'visible_haze'` for the repo's original behaviour.
- Random full-resolution 256 px crops instead of resizing whole frames to 256 px, so the small figures stay visible.
- Output and target are min-max normalized per image before MS-SSIM (the repo normalizes only the output, per batch),
  because the clear target at night is very dark.

**Thermal registration:** thermal is 640x512 and visible 1280x720. `annotations.json` gives the same mapping on all
10 clips, `x_t = x_v / 2`, `y_t = y_v / 2 + 76`, so the visible view is the centre of the thermal view. Thermal rows
76 to 436 are cropped and upscaled 2x. A plain resize to 1280x720 would misalign the two images.

**Outputs** (saved to `DRIVE_OUT` on Colab, so they survive a disconnect):
- `out/<METHOD>_forest_<clip>.mp4`: fine-tuned model. `out/rfn_nest_forest_<clip>.mp4`: published weights, for comparison (`WRITE_BASELINE`).
- `data/forest_<clip>/rgb` + `ir` + `clear`: registered sources and the fog-free reference, in the layout `FUSION_EVALUATION.ipynb` expects.
- `checkpoints/<METHOD>.model` (RFN blocks and NestFuse weights) and `checkpoints/<METHOD>_history.csv`.

To score locally, copy `out/*` to `fusion/out/` and `data/*` to `fusion/data/`, then run `FUSION_EVALUATION.ipynb`
with `SOURCE = 'forest_<clip>'`.

In [ ]:
DATASET = 'forest_batch_1'
TRAIN_CLIPS = ['02_night-dark_low-trunks-north', '03_predawn_oblique-orbit', '05_morning_fast-pass-south',
               '06_midday_nadir-rotate', '08_overcast_hover-southwest', '09_golden_far-west-pullback']
VAL_CLIPS = ['04_sunrise_far-high-south']
TEST_CLIPS = ['01_night-moon_nadir-hover', '07_afternoon_steep-push', '10_dusk_high-orbit']

# Colab only
DRIVE_DATA = '/content/drive/MyDrive/Vidhance Extended View/Blender renders/forest_batch_1.zip'   # .zip, or the forest_batch_1 folder
DRIVE_OUT = '/content/drive/MyDrive/Vidhance Extended View/models_output/rfn_nest_blender'
OUT_ROOT = None               # None -> DRIVE_OUT on Colab, fusion/ when run locally

# Fine-tuning
RETRAIN = False               # False = reuse a finished fine-tuned checkpoint of this METHOD, so re-running skips training
VI_TARGET = 'visible_clear'   # loss target: 'visible_clear' or 'visible_haze' (the repo's behaviour)
EPOCHS = 5                    # the first run stopped improving after epoch 4
LR = 2e-5                     # RFN blocks; the repo uses 1e-4 from scratch, lower because we start from trained weights
TRAIN_DECODER = False         # v3: decoder frozen again (v2 trained it); True = also fine-tune it (the encoder stays frozen)
DECODER_LR = 1e-5             # lower than LR: the decoder is shared by every fusion and was trained on far more data
BATCH_SIZE = 8
CROP = 256                    # training crop size, multiple of 16
TRAIN_STRIDE = 3              # use every n-th frame (neighbouring frames are nearly identical); ~510 frames, ~1.4 GB RAM
CROPS_PER_FRAME = 2           # random crops per frame, per epoch
VAL_STRIDE = 3                # 100 validation frames instead of 30, so the best-epoch choice is less noisy
VAL_CROP = 512
ALPHA, W_IR, W_VI = 2000, 6.0, 3.0   # published: 700, 6.0, 3.0. ALPHA raised so the detail term (vs the fog-free frame)
                                     # is ~80% of the starting loss instead of ~65%; W_IR / W_VI kept, see the notes above
W_FEA = [1, 10, 100, 1000]           # feature-loss weight of each encoder scale, from train_fusionnet.py
SEED = 42

# Output videos
METHOD = 'rfn_nest_blender_v3'   # output file: <METHOD>_forest_<clip>.mp4; one name per version, see the table above
WRITE_BASELINE = False        # also fuse the test clips with the published weights (~50 min on a T4; the first run made them)
EXPORT_EVAL_SOURCES = False   # v1 and v2 already wrote rgb/ + ir/ + clear/ (~2 GB); True if fusion/data/forest_* is missing
NORMALIZE_PER_FRAME = True    # same as rfn_nest.ipynb; set False if the video flickers
MAX_FRAMES = None             # None = whole clip, or an int for a quick look
FPS = 30.0
SCORE_STRIDE = 10             # score every n-th output frame in the summary table

In [ ]:
import os, sys, csv, random, shutil, subprocess, time, zipfile, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np, cv2, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not torch.cuda.is_available():
        raise RuntimeError('No GPU: choose Runtime > Change runtime type > T4 GPU, then run all cells again.')
    VENDOR = Path('/content/_vendor')
    DATA_ROOT = Path('/content/data') / DATASET
    root = Path(OUT_ROOT or DRIVE_OUT)
else:
    here = Path.cwd().resolve()
    fusion = next((d / 'fusion' for d in [here, *here.parents] if (d / 'fusion' / 'data').is_dir()),
                  next((d for d in [here, *here.parents] if d.name == 'fusion' and (d / 'data').is_dir()), None))
    if fusion is None:
        raise FileNotFoundError('Could not find fusion/data from %s. Start Jupyter inside the repo.' % here)
    VENDOR = fusion / 'methods' / '_vendor'
    DATA_ROOT = fusion / 'data' / DATASET
    root = Path(OUT_ROOT) if OUT_ROOT else fusion

OUT_DIR, EVAL_DATA_DIR, CKPT_DIR = root / 'out', root / 'data', root / ('checkpoints' if IN_COLAB or OUT_ROOT else 'out/checkpoints')
for d in (OUT_DIR, EVAL_DATA_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)
FINETUNED_CKPT = CKPT_DIR / (METHOD + '.model')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu')
print('outputs:', root)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
cv2.setNumThreads(1)                                # the frame loader below runs its own threads

## Dataset

Copies the data from Drive to the Colab disk (first run, or when the copy there is incomplete) and checks the split.

In [ ]:
STREAMS = ('visible_haze', 'visible_clear', 'thermal_haze')

def copy_is_complete():
    return all((DATA_ROOT / c / 'frames' / s).is_dir()
               for c in TRAIN_CLIPS + VAL_CLIPS + TEST_CLIPS for s in STREAMS)

if IN_COLAB and DATA_ROOT.is_dir() and not copy_is_complete():
    print('The copy in %s is incomplete (an interrupted upload or unzip?), replacing it.' % DATA_ROOT)
    shutil.rmtree(DATA_ROOT)

if IN_COLAB and not DATA_ROOT.is_dir():
    src, t0 = Path(DRIVE_DATA), time.time()
    if not src.exists():
        raise FileNotFoundError('%s not found. Upload %s to Drive or change DRIVE_DATA.' % (src, DATASET))
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        print('copying the folder from Drive, this can take a long time for this many files...')
        shutil.copytree(src, DATA_ROOT)
    else:
        with zipfile.ZipFile(src) as zf:
            zf.extractall(DATA_ROOT.parent)
    print('data ready in %.0f s' % (time.time() - t0))

def frame_names(clip):
    d = DATA_ROOT / clip / 'frames'
    names = {s: sorted(p.name for p in (d / s).iterdir() if p.suffix == '.png') for s in STREAMS}
    if not names[STREAMS[0]] or any(names[s] != names[STREAMS[0]] for s in STREAMS):
        raise ValueError('%s: the streams do not hold the same frames' % clip)
    return names[STREAMS[0]]

splits = {'train': TRAIN_CLIPS, 'val': VAL_CLIPS, 'test': TEST_CLIPS}
all_clips = TRAIN_CLIPS + VAL_CLIPS + TEST_CLIPS
if len(set(all_clips)) != len(all_clips):
    raise ValueError('A clip is in more than one split')
FRAMES = {c: frame_names(c) for c in all_clips}
for split, clips in splits.items():
    print('%-5s %d clips, %d frames' % (split, len(clips), sum(len(FRAMES[c]) for c in clips)))

# Thermal to visible registration, from annotations.json: x_t = x_v / 2, y_t = y_v / 2 + 76
VIS_W, VIS_H = 1280, 720
TH_Y0, TH_Y1 = 76, 436

def read_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError('Could not read %s' % path)
    return img

def read_stream(clip, stream, name):
    img = read_gray(DATA_ROOT / clip / 'frames' / stream / name)
    if stream == 'thermal_haze':
        img = cv2.resize(img[TH_Y0:TH_Y1], (VIS_W, VIS_H), interpolation=cv2.INTER_LINEAR)
    return img

def load_frames(clips, stride):
    """Loads every stride-th frame into RAM as uint8 arrays: (thermal, hazy visible, loss target)."""
    jobs = [(c, n) for c in clips for n in FRAMES[c][::stride]]
    def one(job):
        c, n = job
        return read_stream(c, 'thermal_haze', n), read_stream(c, 'visible_haze', n), read_stream(c, VI_TARGET, n)
    with ThreadPoolExecutor(4) as pool:
        return list(tqdm(pool.map(one, jobs), total=len(jobs), desc='loading frames'))

## RFN-Nest code and published checkpoints

In [ ]:
REPO_DIR = VENDOR / 'imagefusion-rfn-nest'
if not REPO_DIR.is_dir():
    VENDOR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/hli1221/imagefusion-rfn-nest.git', str(REPO_DIR)], check=True)
MODELS = REPO_DIR / 'models'
for stem in ('nestfuse', 'rfn_twostage'):
    if not (MODELS / stem).is_dir():
        with zipfile.ZipFile(MODELS / (stem + '.zip')) as zf:
            zf.extractall(MODELS)

sys.path.insert(0, str(REPO_DIR))
with warnings.catch_warnings():
    warnings.simplefilter('ignore')                  # net.py uses `is` with literals (SyntaxWarning)
    from net import NestFuse_light2_nodense, Fusion_network
    from pytorch_msssim import msssim

NB_FILTER = [64, 112, 160, 208, 256]
NEST_CKPT = MODELS / 'nestfuse' / 'nestfuse_gray_1e2.model'
PUBLISHED_RFN_CKPT = MODELS / 'rfn_twostage' / '6.0' / 'Final_epoch_2_alpha_700_wir_6.0_wvi_3.0_ssim_vi.model'

ENCODER = ('conv0.', 'DB1_0.', 'DB2_0.', 'DB3_0.', 'DB4_0.')   # NestFuse encoder layers; the rest is the decoder

def load_nest(state=None):
    """NestFuse, frozen; the published weights unless a fine-tuned state is given."""
    model = NestFuse_light2_nodense(NB_FILTER, 1, 1, deepsupervision=False)
    model.load_state_dict(state if state is not None else torch.load(NEST_CKPT, map_location='cpu'))
    return model.eval().requires_grad_(False).to(DEVICE)

def load_rfn(state):
    model = Fusion_network(NB_FILTER, 'res')
    model.load_state_dict(state)
    return model.to(DEVICE)

def load_models(path):
    """(NestFuse, RFN) for a checkpoint: this notebook's {'rfn', 'nest'} format, or an RFN-only one such as the
    published weights or the first run's rfn_nest_blender.model, which use the published NestFuse."""
    state = torch.load(path, map_location='cpu')
    if 'rfn' in state:
        return load_nest(state['nest']), load_rfn(state['rfn'])
    return load_nest(), load_rfn(state)

def minmax(x):
    """Scales each image of a batch to 0-255."""
    lo = x.amin(dim=(1, 2, 3), keepdim=True)
    hi = x.amax(dim=(1, 2, 3), keepdim=True)
    return (x - lo) / (hi - lo + 1e-5) * 255

def to_batch(arrays):
    return torch.from_numpy(np.stack(arrays)).float()[:, None].to(DEVICE)

## Fine-tuning

The validation loss is measured before the first epoch too, so epoch 0 is the published checkpoint. The checkpoint
with the lowest validation loss is the one kept, and both validation terms are printed, so you can see which one
improves. If no epoch beats epoch 0, the output stays the published weights and the notebook says so.

During training the best checkpoint is written to `<METHOD>.model.partial` and only renamed to `<METHOD>.model` when
training finishes, so an interrupted run is never mistaken for a finished one: the next run starts training again.

In [ ]:
def compute_loss(nest, rfn, ir, vi, target):
    with torch.no_grad():
        en_ir, en_vi = nest.encoder(ir), nest.encoder(vi)
        en_target = en_vi if VI_TARGET == 'visible_haze' else nest.encoder(target)
    fused = rfn(en_ir, en_vi)
    output = nest.decoder_eval(fused)[0]
    detail = ALPHA * (1 - msssim(minmax(output), minmax(target), val_range=255, normalize=True))
    feature = sum(w * F.mse_loss(f, W_IR * a + W_VI * b) for w, f, a, b in zip(W_FEA, fused, en_ir, en_target))
    return detail, feature

def train_batches(data):
    order = np.random.permutation(np.repeat(np.arange(len(data)), CROPS_PER_FRAME))
    for start in range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE):
        crops = ([], [], [])
        for idx in order[start:start + BATCH_SIZE]:
            y = np.random.randint(VIS_H - CROP + 1)
            x = np.random.randint(VIS_W - CROP + 1)
            flip = np.random.random() < 0.5
            for out, img in zip(crops, data[idx]):
                crop = img[y:y + CROP, x:x + CROP]
                out.append(crop[:, ::-1].copy() if flip else crop)
        yield tuple(to_batch(c) for c in crops)

def validate(nest, rfn, data):
    """Mean (detail, feature) loss on a fixed centre crop of every validation frame."""
    rfn.eval()
    y, x = (VIS_H - VAL_CROP) // 2, (VIS_W - VAL_CROP) // 2
    sums, step = np.zeros(2), max(1, BATCH_SIZE // 2)
    with torch.no_grad():
        for start in range(0, len(data), step):
            part = data[start:start + step]
            batch = [to_batch([f[s][y:y + VAL_CROP, x:x + VAL_CROP] for f in part]) for s in range(3)]
            detail, feature = compute_loss(nest, rfn, *batch)
            sums += np.array([detail.item(), feature.item()]) * len(part)
    return sums / len(data)

if FINETUNED_CKPT.exists() and not RETRAIN:
    print('Reusing', FINETUNED_CKPT, '(set RETRAIN = True to train again)')
else:
    train_data, val_data = load_frames(TRAIN_CLIPS, TRAIN_STRIDE), load_frames(VAL_CLIPS, VAL_STRIDE)
    nest, rfn = load_models(PUBLISHED_RFN_CKPT)
    groups = [dict(params=list(rfn.parameters()), lr=LR)]
    if TRAIN_DECODER:
        decoder = [p for name, p in nest.named_parameters() if not name.startswith(ENCODER)]
        for p in decoder:
            p.requires_grad_(True)
        groups.append(dict(params=decoder, lr=DECODER_LR))
    print('training %d RFN + %d decoder parameters' % (sum(p.numel() for p in groups[0]['params']),
                                                        sum(p.numel() for g in groups[1:] for p in g['params'])))
    optimizer = torch.optim.Adam(groups)
    partial = FINETUNED_CKPT.with_name(FINETUNED_CKPT.name + '.partial')
    save = lambda: torch.save({'rfn': rfn.state_dict(), 'nest': nest.state_dict()}, partial)

    val_detail, val_feature = validate(nest, rfn, val_data)
    best_val, best_epoch = val_detail + val_feature, 0
    save()
    history = [dict(epoch=0, detail=None, feature=None, val_detail=val_detail, val_feature=val_feature, val=best_val)]
    print('epoch  0  published weights                 val %.2f (detail %.2f, feature %.2f)'
          % (best_val, val_detail, val_feature))
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        rfn.train()
        sums, steps = np.zeros(2), 0
        for ir, vi, target in tqdm(train_batches(train_data), leave=False, desc='epoch %d' % epoch,
                                   total=len(train_data) * CROPS_PER_FRAME // BATCH_SIZE):
            detail, feature = compute_loss(nest, rfn, ir, vi, target)
            optimizer.zero_grad()
            (detail + feature).backward()
            optimizer.step()
            sums += detail.item(), feature.item()
            steps += 1
        val_detail, val_feature = validate(nest, rfn, val_data)
        val, mark = val_detail + val_feature, ''
        if val < best_val:
            best_val, best_epoch, mark = val, epoch, '  saved'
            save()
        history.append(dict(epoch=epoch, detail=sums[0] / steps, feature=sums[1] / steps,
                            val_detail=val_detail, val_feature=val_feature, val=val))
        print('epoch %2d  detail %.2f  feature %.2f  val %.2f (detail %.2f, feature %.2f)  (%.0f s)%s'
              % (epoch, sums[0] / steps, sums[1] / steps, val, val_detail, val_feature, time.time() - t0, mark))
    os.replace(partial, FINETUNED_CKPT)               # only now does the checkpoint count as finished
    with open(CKPT_DIR / (METHOD + '_history.csv'), 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    if best_epoch == 0:
        print('WARNING: no epoch beat the published weights on validation, so %s holds the published weights.' % METHOD)
    else:
        print('best epoch %d: val %.2f (published weights: %.2f)' % (best_epoch, best_val, history[0]['val']))
    del nest, rfn, optimizer, train_data, val_data
    torch.cuda.empty_cache()

## Fuse the test clips

Whole frames are fused at once, because 720p fits on the T4. `rfn_nest.ipynb` splits large frames into four tiles for
CPU memory, which is not needed here and can leave seams.

The table scores every `SCORE_STRIDE`-th frame with MS-SSIM against the fog-free render, which is the reference only
this dataset has, and against the hazy input. It is a quick check: the full metrics are in `FUSION_EVALUATION.ipynb`.

In [ ]:
@torch.no_grad()
def fuse_frame(nest, rfn, ir, vi):
    fused = nest.decoder_eval(rfn(nest.encoder(to_batch([ir])), nest.encoder(to_batch([vi]))))[0]
    out = fused[0, 0].float().cpu().numpy()
    if NORMALIZE_PER_FRAME:
        out = (out - out.min()) / (out.max() - out.min() + 1e-5) * 255
    return np.clip(out, 0, 255).astype(np.uint8)

@torch.no_grad()
def similarity(a, b):
    return msssim(minmax(to_batch([a])), minmax(to_batch([b])), val_range=255, normalize=True).item()

def make_writer(path):
    for codec in ('avc1', 'mp4v'):                   # H.264 if this OpenCV build has it (a log line for avc1 is harmless)
        writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*codec), FPS, (VIS_W, VIS_H))
        if writer.isOpened():
            return writer
    raise RuntimeError('OpenCV could not open a video writer for %s' % path)

def fuse_clip(nest, rfn, method, clip):
    """Writes <method>_forest_<clip>.mp4 and returns the summary row and the middle frame."""
    rfn.eval()
    names = FRAMES[clip][:MAX_FRAMES]
    out_path = OUT_DIR / ('%s_forest_%s.mp4' % (method, clip))
    tmp_path = OUT_DIR / ('.%s_forest_%s.partial.mp4' % (method, clip))   # renamed only when finished
    writer, clear_scores, haze_scores, middle, t0 = make_writer(tmp_path), [], [], None, time.time()
    try:
        for i, name in enumerate(tqdm(names, desc='%s %s' % (method, clip), leave=False)):
            ir, vi = read_stream(clip, 'thermal_haze', name), read_stream(clip, 'visible_haze', name)
            fused = fuse_frame(nest, rfn, ir, vi)
            writer.write(cv2.cvtColor(fused, cv2.COLOR_GRAY2BGR))
            if i % SCORE_STRIDE == 0:
                clear_scores.append(similarity(fused, read_stream(clip, 'visible_clear', name)))
                haze_scores.append(similarity(fused, vi))
            if i == len(names) // 2:
                middle = fused
    finally:
        writer.release()
    os.replace(tmp_path, out_path)
    sec = time.time() - t0
    print('%d frames in %.0f s (%.1f fps) -> %s' % (len(names), sec, len(names) / sec, out_path.name))
    return dict(clip=clip, method=method, ms_ssim_clear=np.mean(clear_scores), ms_ssim_haze=np.mean(haze_scores)), middle

models = {METHOD: FINETUNED_CKPT}
if WRITE_BASELINE:
    models['rfn_nest'] = PUBLISHED_RFN_CKPT

rows, middles = [], {}
for method, ckpt in models.items():
    nest, rfn = load_models(ckpt)
    for clip in TEST_CLIPS:
        row, middles[clip, method] = fuse_clip(nest, rfn, method, clip)
        rows.append(row)
    del nest, rfn

print('\n%-32s %-18s %14s %13s' % ('test clip', 'method', 'vs fog-free', 'vs hazy'))
for r in rows:
    print('%-32s %-18s %14.3f %13.3f' % (r['clip'], r['method'], r['ms_ssim_clear'], r['ms_ssim_haze']))
if torch.cuda.is_available():
    print('\npeak GPU memory: %.1f GB' % (torch.cuda.max_memory_allocated() / 1e9))

In [ ]:
# Middle frame of each test clip
columns = ['visible_haze', 'thermal_haze', *models, 'visible_clear']
fig, axes = plt.subplots(len(TEST_CLIPS), len(columns), figsize=(4 * len(columns), 2.4 * len(TEST_CLIPS)), squeeze=False)
for row_axes, clip in zip(axes, TEST_CLIPS):
    names = FRAMES[clip][:MAX_FRAMES]
    name = names[len(names) // 2]
    for ax, col in zip(row_axes, columns):
        ax.imshow(middles[clip, col] if col in models else read_stream(clip, col, name), cmap='gray', vmin=0, vmax=255)
        ax.set_title(col, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    row_axes[0].set_ylabel(clip[:2], rotation=0, labelpad=12)
fig.tight_layout()
plt.show()

## Export the sources for `FUSION_EVALUATION.ipynb`

That notebook needs `data/<source>/rgb` and `ir` at the size of the video, with the same frames in the same order.
`rgb` is `visible_haze`, the actual input. `ir` is the registered thermal. `clear` is `visible_clear`, the fog-free
reference for its SSIM_ref and PSNR_ref scores.

In [ ]:
if EXPORT_EVAL_SOURCES:
    for clip in TEST_CLIPS:
        dst = EVAL_DATA_DIR / ('forest_' + clip)
        for sub in ('rgb', 'ir', 'clear'):
            (dst / sub).mkdir(parents=True, exist_ok=True)
        for name in tqdm(FRAMES[clip][:MAX_FRAMES], desc=clip, leave=False):
            shutil.copyfile(DATA_ROOT / clip / 'frames' / 'visible_haze' / name, dst / 'rgb' / name)
            shutil.copyfile(DATA_ROOT / clip / 'frames' / 'visible_clear' / name, dst / 'clear' / name)
            cv2.imwrite(str(dst / 'ir' / name), read_stream(clip, 'thermal_haze', name))
        print('exported', dst)

print('\nDone. Videos are in', OUT_DIR)
print('Score them with FUSION_EVALUATION.ipynb, SOURCE =', ['forest_' + c for c in TEST_CLIPS])